# CoalGameRec local run notebook — Mac M4 Pro / 48GB RAM

This notebook runs an end-to-end **local executable prototype** of the CoalGameRec case-study pipeline:

1. load MovieLens-1M (automatic download);
2. optional Amazon Books 2018 loader if you provide `Books_5.json.gz`;
3. convert ratings to implicit positives;
4. build a temporal leave-one-out split;
5. train a small frozen BPR-MF backbone on Apple Silicon (`mps` if available);
6. cache full-catalogue base scores;
7. compute train-only item vectors;
8. compute post-hoc Shapley attributions on validation relevance;
9. apply fixed post-hoc reranking;
10. report HitRate@K and NDCG@K.

**Important:** this is a Mac-local implementation/prototype. It is not the validated official HCCF port required for confirmatory preregistration. The HCCF port, `PORT.md`, validation logs, lockfile/container, ethics determination, and external preregistration remain required real artifacts.

In [ ]:
# If needed, run once in your environment:
# %pip install -r ../requirements.txt

from pathlib import Path
import sys, json, platform

# Robust path setup whether the kernel cwd is code/ or code/notebooks/
CWD = Path.cwd().resolve()
CODE_DIR = CWD if (CWD / 'coalgamerec').exists() else CWD.parent
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import numpy as np
import pandas as pd
import torch

from coalgamerec.data import load_movielens_1m, load_amazon_books_2018, preprocess_temporal_loo, item_user_vectors
from coalgamerec.models import TrainConfig, train_bprmf, cache_full_scores, pick_device
from coalgamerec.metrics import evaluate
from coalgamerec.attribution import compute_shapley_for_users
from coalgamerec.rerank import rerank_all
from coalgamerec.validation import assert_item_vector_isolation, assert_rerank_nonzero, assert_shapley_shapes

print(platform.platform())
print('torch', torch.__version__)
print('mps available:', torch.backends.mps.is_available() if hasattr(torch.backends, 'mps') else False)
print('device:', pick_device('auto'))

## Configuration

Defaults are set for a quick Mac M4 Pro feasibility run. For a more complete MovieLens run, increase `SAMPLE_USERS`, `EPOCHS`, and `SHAPLEY_USERS`.

In [ ]:
ROOT = CODE_DIR
DATA_RAW = ROOT / 'data' / 'raw'
RESULTS = ROOT / 'results' / 'mac_run'
RESULTS.mkdir(parents=True, exist_ok=True)

# Quick local defaults. Set SAMPLE_USERS=None for full MovieLens-1M.
SAMPLE_USERS = 2000
EPOCHS = 8
DIM = 64
BATCH_SIZE = 4096
SEED = 42

# Shapley is the expensive step. Use None for all users after feasibility testing.
SHAPLEY_USERS = 500
M_PERMUTATIONS = 64  # preregistration design says 128; 64 is faster for local smoke runs.
LAMBDA_ATTR = 0.10
KS = (5, 10, 20)

cfg = dict(SAMPLE_USERS=SAMPLE_USERS, EPOCHS=EPOCHS, DIM=DIM, BATCH_SIZE=BATCH_SIZE, SEED=SEED, SHAPLEY_USERS=SHAPLEY_USERS, M_PERMUTATIONS=M_PERMUTATIONS, LAMBDA_ATTR=LAMBDA_ATTR)
print(json.dumps(cfg, indent=2))

## Load MovieLens-1M and build temporal leave-one-out split

In [ ]:
ratings = load_movielens_1m(DATA_RAW)
print(ratings.head())
print('raw rows:', len(ratings), 'users:', ratings.user_raw.nunique(), 'items:', ratings.item_raw.nunique())

split, stats = preprocess_temporal_loo(ratings, name='ml1m', sample_users=SAMPLE_USERS, sample_seed=SEED)
print(json.dumps(stats, indent=2, default=str))
split.train.head()

## Optional: Amazon Books 2018 loader

Download `Books_5.json.gz` manually from the UCSD Amazon Reviews 2018 page, then set `AMAZON_BOOKS_5` below. The full file is very large; start with `max_rows` for a feasibility spike.

In [ ]:
RUN_AMAZON = False
AMAZON_BOOKS_5 = DATA_RAW / 'Books_5.json.gz'

if RUN_AMAZON:
    amazon = load_amazon_books_2018(AMAZON_BOOKS_5, max_rows=2_000_000)
    amazon_split, amazon_stats = preprocess_temporal_loo(amazon, name='amazon_books_2018_sample', sample_users=50000, sample_seed=SEED)
    print(json.dumps(amazon_stats, indent=2, default=str))


## Train frozen backbone

The notebook uses a small BPR-MF backbone so the full pipeline runs on a Mac. Replace this with the validated HCCF port once that artifact exists.

In [ ]:
train_cfg = TrainConfig(dim=DIM, epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED, device='auto')
model = train_bprmf(split.train, split.n_users, split.n_items, train_cfg, verbose=True)
base_scores = cache_full_scores(model, split.n_users, batch_size=256)
print(base_scores.shape, base_scores.dtype)

## Build train-only item vectors and evaluate base model

In [ ]:
X_items = item_user_vectors(split.train_csr)
print('item vectors:', X_items.shape, 'nnz:', X_items.nnz, 'density:', X_items.nnz / (X_items.shape[0] * X_items.shape[1]))

base_summary, base_per_user = evaluate(base_scores, split, X_items, ks=KS)
print('item-vector isolation:', assert_item_vector_isolation(split))
pd.Series(base_summary, name='base').to_frame()

## Compute Shapley attributions using validation relevance

This is the expensive part. The notebook defaults to the first 500 users and `M=64` for speed. For a closer preregistration-like run, set `SHAPLEY_USERS=None` and `M_PERMUTATIONS=128`.

In [ ]:
shapley = compute_shapley_for_users(
    split, base_scores, X_items,
    max_users=SHAPLEY_USERS,
    m=M_PERMUTATIONS,
    exact_threshold=8,
    seed=SEED,
    alpha=0.70, beta=0.30, lambda_pref=0.20, lambda_attr_value=0.10,
)
# Users without Shapley in quick mode get zero weights so the notebook can still evaluate all users.
print('computed users:', len(shapley))
print('shapley shapes:', assert_shapley_shapes(split, shapley))
first_u = next(iter(shapley))
print(first_u, shapley[first_u][:10])

## Rerank and evaluate attribution families

In [ ]:
print('rerank nonzero:', assert_rerank_nonzero(split, base_scores, X_items, family='uniform'))
families = ['uniform', 'additive-pref', 'attention', 'heuristic-pop', 'shapley-mc']
rows = []
for fam in families:
    scores = rerank_all(base_scores, split, X_items, fam, shapley_by_user=shapley, lambda_attr=LAMBDA_ATTR)
    summary, _ = evaluate(scores, split, X_items, ks=KS)
    summary['family'] = fam
    rows.append(summary)
results = pd.DataFrame(rows).set_index('family')
results.to_csv(RESULTS / 'ml1m_mac_local_results.csv')
results

## Reranking strength sensitivity

In [ ]:
sens_rows = []
for lam in [0.05, 0.10, 0.20]:
    for fam in ['uniform', 'additive-pref', 'shapley-mc']:
        scores = rerank_all(base_scores, split, X_items, fam, shapley_by_user=shapley, lambda_attr=lam)
        summary, _ = evaluate(scores, split, X_items, ks=KS)
        summary.update({'family': fam, 'lambda_attr': lam})
        sens_rows.append(summary)
sensitivity = pd.DataFrame(sens_rows)
sensitivity.to_csv(RESULTS / 'ml1m_lambda_sensitivity.csv', index=False)
sensitivity

## Save run manifest

In [ ]:
manifest = {
    'note': 'Mac-local CoalGameRec prototype; not confirmatory HCCF preregistration run',
    'config': cfg,
    'dataset_stats': stats,
    'torch': torch.__version__,
    'device': str(pick_device('auto')),
}
(RESULTS / 'manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
print('wrote', RESULTS)